In [1]:
# import the necessary libraries to execute this code
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.model_selection import RandomizedSearchCV as RSCV
import pickle
from lightgbm import LGBMRegressor

# NESTED_CV for the reduced feature model (LGBM)

In [2]:
datafile = "C:/chem papers/Dataset_17_feat.xlsx"
df = pd.read_excel(datafile)
          
model = LGBMRegressor(random_state=4)
p_grid ={"n_estimators":[100,150,200,250,300,400,500,600],
        'boosting_type': ['gbdt', 'dart', 'goss'],
        'num_leaves':[16,32,64,128,256],
        'learning_rate':[0.1,0.01,0.001,0.0001],
        'min_child_weight': [0.001,0.01,0.1,1.0,10.0],
        'subsample': [0.4,0.6,0.8,1.0],
        'min_child_samples':[2,10,20,40,100],
        'reg_alpha': [0, 0.005, 0.01, 0.015],
        'reg_lambda': [0, 0.005, 0.01, 0.015]}
        
X = df.drop(['Experimental_index','DP_Group','Release'],axis='columns')
stdScale = StandardScaler().fit(X)
X=stdScale.transform(X)
Y = df['Release']
G = df['DP_Group']
E = df['Experimental_index']
T = df['Time']    

In [3]:
NUM_TRIALS = 10

itr_number = [] # create new empty list for itr number 
outer_results = []
inner_results = []
model_params = []
G_test_list = []
y_test_list = []
E_test_list = []
T_test_list = []
pred_list = []

for i in range(NUM_TRIALS): #configure the cross-validation procedure - outer loop (test set) 
    
    cv_outer = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=i) #hold back 20% of the groups for test set
    
    # split data using GSS
    for train_index, test_index in cv_outer.split(X, Y, G):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = Y[train_index], Y[test_index]
        G_train, G_test = G[train_index], G[test_index]
        E_train, E_test = E[train_index], E[test_index]
        T_train, T_test = T[train_index], T[test_index]

        # store test set information
        G_test = np.array(G_test) #prevents index from being brought from dataframe
        G_test_list.append(G_test)
        E_test = np.array(E_test) #prevents index from being brought from dataframe
        E_test_list.append(E_test)
        T_test = np.array(T_test) #prevents index from being brought from dataframe
        T_test_list.append(T_test)
        y_test = np.array(y_test) #prevents index from being brought from dataframe
        y_test_list.append(y_test)

        # configure the cross-validation procedure - inner loop (validation set/HP optimization)
        cv_inner = GroupKFold(n_splits=10) #should be 10 fold group split for inner loop

        # define search space
        search = RSCV(model, p_grid, n_iter=100, verbose=0, scoring='neg_mean_absolute_error', n_jobs= 6, cv=cv_inner, refit=True) # should be 100

        # execute search
        result = search.fit(X_train, y_train, groups=G_train)

        # get the best performing model fit on the whole training set
        best_model = result.best_estimator_
        
        # get the score for the best performing model and store
        best_score = abs(result.best_score_)
        inner_results.append(best_score)

        # evaluate model on the hold out dataset
        yhat = np.round(best_model.predict(X_test), 3)

        
        # store drug release predictions
        pred_list.append(yhat)

        # evaluate the model
        acc = mean_absolute_error(y_test, yhat)

        # store the result
        itr_number.append(i+1)
        outer_results.append(acc)
        model_params.append(result.best_params_)

        # report progress at end of each inner loop
        print('\n################################################################\n\nSTATUS REPORT:') 
        print('Iteration '+str(i+1)+' of '+str(NUM_TRIALS)+' completed') 
        print('Test_Score: %.3f, Best_Valid_Score: %.3f, \n\nBest_Model_Params: \n%s' % (acc, best_score, result.best_params_))
        print("\n################################################################\n ")


C:\Users\prajw\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\prajw\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^

[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001430 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1214
[LightGBM] [Info] Number of data points in the train set: 3463, number of used features: 17
[LightGBM] [Info] Using GOSS
[LightGBM] [Info] Start training from score 0.476951


C:\Users\prajw\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.

################################################################

STATUS REPORT:
Iteration 1 of 10 completed
Test_Score: 0.084, Best_Valid_Score: 0.131, 

Best_Model_Params: 
{'subsample': 0.6, 'reg_lambda': 0, 'reg_alpha': 0.015, 'num_leaves': 64, 'n_estimators': 250, 'min_child_weight': 0.1, 'min_child_samples': 10, 'learning_rate': 0.1, 'boosting_type': 'goss'}

################################################################
 
[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_stra

C:\Users\prajw\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


...

C:\Users\prajw\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.

################################################################

STATUS REPORT:
Iteration 8 of 10 completed
Test_Score: 0.099, Best_Valid_Score: 0.134, 

Best_Model_Params: 
{'subsample': 0.4, 'reg_lambda': 0, 'reg_alpha': 0, 'num_leaves': 16, 'n_estimators': 150, 'min_child_weight': 0.1, 'min_child_samples': 10, 'learning_rate': 0.1, 'boosting_type': 'goss'}

################################################################
 
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001690 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1136
[LightGBM] [Info] Number of data points in the train set: 3101, number of used features: 17
[LightGBM] [Info] Start training from score 0.460015
[LightGBM] [Wa

C:\Users\prajw\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



################################################################

STATUS REPORT:
Iteration 9 of 10 completed
Test_Score: 0.156, Best_Valid_Score: 0.121, 

Best_Model_Params: 
{'subsample': 0.4, 'reg_lambda': 0.005, 'reg_alpha': 0.005, 'num_leaves': 256, 'n_estimators': 150, 'min_child_weight': 0.001, 'min_child_samples': 40, 'learning_rate': 0.1, 'boosting_type': 'dart'}

################################################################
 
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000539 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1071
[LightGBM] [Info] Number of data points in the train set: 2984, number of used features: 17
[LightGBM] [Info] Start training from score 0.468076
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best ga

C:\Users\prajw\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [7]:
#create dataframe with results of nested CV
list_of_tuples = list(zip(itr_number, inner_results, outer_results, model_params, G_test_list, E_test_list, T_test_list, y_test_list, pred_list))
CV_dataset = pd.DataFrame(list_of_tuples, columns = ['Iter', 'Valid Score', 'Test Score', 'Model Parms', 'DP_Groups', "Experimental Index", "Time", 'Experimental_Release', 'Predicted_Release'])
CV_dataset['Score_difference'] = abs(CV_dataset['Valid Score'] - CV_dataset['Test Score']) #Groupby dataframe model iterations that best fit the data (i.e., validitaion <= test)
CV_dataset.sort_values(by=['Score_difference', 'Test Score'], ascending=True, inplace=True) 
CV_dataset = CV_dataset.reset_index(drop=True) # Reset index of dataframe
CV_dataset.to_pickle("C:/chem papers/TRAINED_MODELS/17_feat_LGBM_model.pkl", compression='infer', protocol=5, storage_options=None) # save dataframe as pickle file
CV_dataset.describe()

,Iter,Valid Score,Test Score,Score_difference
count,10.00000,10.000000,10.000000,10.000000
mean,5.50000,0.126389,0.118627,0.025207
std,3.02765,0.006633,0.026606,0.019069
min,1.00000,0.115659,0.083616,0.001616
25%,3.25000,0.121436,0.100753,0.007851
50%,5.50000,0.128262,0.119103,0.026606
75%,7.75000,0.130975,0.126253,0.042994
max,10.00000,0.134590,0.166017,0.048205


In [9]:
best_model_params = CV_dataset.iloc[0,3] # assign the best model paramaters
LGBM_15 = model.set_params(**best_model_params) # set params from the best model
LGBM_15 = LGBM_15.fit(X, Y)
with open('C:/chem papers/TRAINED_MODELS/17_feat_LGBM_model.pkl', 'wb') as file: # Save the Model to pickle file
          pickle.dump(LGBM_15, file)
LGBM_15

[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000580 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1300
[LightGBM] [Info] Number of data points in the train set: 3783, number of used features: 17
[LightGBM] [Info] Using GOSS
[LightGBM] [Info] Start training from score 0.464868


LGBMRegressor(boosting_type='goss', learning_rate=0.01, min_child_samples=2,
              min_child_weight=1.0, n_estimators=500, num_leaves=16,
              random_state=4, reg_alpha=0, reg_lambda=0.015, subsample=0.4)